# **DLO-JZ Optimisation de l'apprentissage**
<img src="./images/optimisation.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-success">
    
## Objet des notebooks

Le but de ces trois *notebooks* est d'optimiser un code d'apprentissage d'un modèle *Resnet-50* sur *Imagenet* pour Jean Zay en implémentant :
* **TP2.1** : l'optimisation du *Dataloader*
* **TP2.2** : la DDP (*Distributed Data Parallelism*)
* **TP2.3** : la DDP et le problème des paramètres non utilisés


Les cellules dans ce *notebook* ne sont pas prévues pour être modifiées, sauf rares exceptions indiquées dans les commentaires. Les TP se feront en modifiant les codes `dlojz1_X.py`.

Les directives de modification seront marquées par l'étiquette suivante <div class="alert alert-block alert-warning">**TODO**</div>
Des solutions sont présentes dans le répertoire `solutions/`.

*Notebook rédigé par l'équipe assistance IA de l'IDRIS, mai 2026.*

</div>

### **Environnement de calcul**

Les fonctions *python* de gestion de queue SLURM développées par l'IDRIS et les fonctions dédiées à la formation DLO-JZ sont à importer.

Le module d'environnement pour les *jobs* et la taille des images sont fixés pour ce *notebook*.
<div class="alert alert-block alert-warning">
    
**TODO :** choisir un pseudonyme (maximum 5 caractères) pour vous différencier dans la queue SLURM pendant la formation.

</div>

In [ ]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import controle_technique, compare, comm_profiler, turbo_profiler, BatchNorm_view, metric_compute_log
MODULE = 'pytorch-gpu/py3/2.8.0'
image_size = 224
account = 'for@a100'
name = 'pseudo'   ## Pseudonyme à choisir
!mkdir -p checkpoints # Création d'un répertoire `checkpoints/` si cela n'a pas déjà été fait.

### **Gestion de la queue SLURM**

Pour afficher vos jobs dans la queue SLURM :

In [ ]:
display_slurm_queue(name)

**Remarque**: cette fonction sera utilisée plusieurs fois dans ce *notebook*. Elle permet d'afficher la queue de manière dynamique, rafraichie toutes les 5 secondes. Elle ne s'arrête que lorsque la queue est vide. Si vous désirez reprendre la main sur le *notebook*, il vous suffira d'arrêter manuellement la cellule avec le bouton *stop*. Cela n'a bien sûr aucun impact sur les *jobs* soumis.

Si vous voulez retirer TOUS vos *jobs* de la queue SLURM, décommenter et exécuter la cellule suivante :

In [ ]:
#!scancel -u $USER

Si vous voulez retirer UN de vos *jobs* de la queue SLURM, décommenter, compléter et exécuter la cellule suivante :

In [ ]:
#!scancel <jobid>

--------------

### Différence entre deux scripts

Pour comparer son code avec les solutions mises à disposition, la fonction suivante permet d'afficher une page HTML contenant un différentiel de fichiers texte.

In [ ]:
s1 = "./dlojz1_2.py"
s2 = "./solutions/dlojz1_2.py"
compare(s1, s2)

Voir le résultat du différentiel de fichiers sur la page suivante (attention au spoil !) :

[compare.html](compare.html)

----------------------

<div class="alert alert-block alert-success">

# TP2.2 : Distribution - Parallélisme de données

Voir la [documentation de l'IDRIS](http://www.idris.fr/docs/jean-zay/intelligence_artificielle/distribution_parallelisme/data-parallelism-pytorch).

</div>
<div class="alert alert-block alert-warning">

**TODO** : dans le script [dlojz1_2.py](./dlojz1_2.py) :
* Importer les librairies liées à la distribution et au *Data Parallelism*.

* Configurer et initialiser l'environnement parallèle.

* Associer le bon GPU alloué au *process* actif.

* Basculer le modèle en mode *DistributedDataParallelism* pour qu'il soit dupliqué sur les différents GPU.

* Définir les *samplers* distribués `train_sampler` et `val_sampler` et les utiliser dans `train_loader` et `val_loader` respectivement. ***Attention***, le *shuffling* devra être délégué aux samplers.
    
* Au tout début de la boucle d'apprentissage, indiquer au *sampler* l'*epoch* en cours afin d'obtenir un *shuffling* différent à chaque *epoch*.

</div>

## Garage - Mise à niveau

In [ ]:
image_size = 224
bs_optim = 512

## Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_2.py -b {bs_optim} --image-size {image_size} --test --chkpt' 
n_gpu = 4
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

<div class="alert alert-block alert-warning">

### **Quizz**
L'éxécution étant assez longue, un quizz vous attend : [Quizz TP1.2](https://www.deepmama.com/quizz/dlojz_quizz5.html)

</div>

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-info">

### Communications
#### Découverte de comm_profiler
Pour ce TP, nous avons implémenté un profiler maison léger `comm_profiler` basé sur les traces de DEBUG de NCCL pour visualiser la quantité et le type de communications collectives échangées pendant une boucle d'apprentissage distribuée sur plusieurs GPU.

**À noter :** dans le script python [dlojz1_2.py](./dlojz1_2.py) les variables de trace de *DEBUG* *NCCL* sont configurées comme suit :

```python
if __name__ == '__main__':
    
    os.environ["NCCL_DEBUG"] = "INFO"
    os.environ["NCCL_DEBUG_SUBSYS"] = "INIT,COLL"
    # display info
    ...
```

</div>

In [ ]:
dfplot = comm_profiler(jobid, n_display=65)

<div class="alert alert-block alert-info">

[Extrait de la documentation pytorch : ](https://pytorch.org/docs/stable/notes/ddp.html#internal-design)

> Each DDP process creates a local `Reducer`, which will take care of the gradients synchronization during the backward pass. To improve communication efficiency, the `Reducer` organizes parameter gradients into **buckets**, and reduces one bucket at a time. **Bucket size** can be configured by setting the bucket_cap_mb argument in DDP constructor. The mapping from parameter gradients to buckets is determined at the construction time, based on the bucket size limit and parameter sizes. 

</div>
<img src="./images/buckets1.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-info">

## Synchronized Metrics Communications 

D'après la [documentation de TorchMetrics](https://lightning.ai/docs/torchmetrics/stable/pages/overview.html):

> TorchMetrics is a Metrics API created for easy metric development and usage in PyTorch and PyTorch Lightning. It is rigorously tested for all edge cases and includes a growing list of common metric implementations.
>
> The metrics API provides `update()`, `compute()`, `reset()` functions to the user. The metric base class inherits `torch.nn.Module` which allows us to call `metric(...)` directly. The `forward()` method of the base Metric class serves the dual purpose of calling `update()` on its input and simultaneously returning the value of the metric over the provided input.
>
> These metrics work with **DDP** in PyTorch and PyTorch Lightning by default. When `.compute()` is called in distributed mode, the internal state of each metric is synced and reduced across each process, so that the logic present in `.compute()` is applied to state information from all processes.

Dans le test précedent, nous appliquions un `compute()` des *metrics* toutes les **100 itérations** pendant l'apprentissage et à la fin de chaque validation. Nous ne pouvions donc voir les communications de synchronisation des *metrics* qu'à la fin de la validation.

**Dans le test suivant,** nous appliquerons un `compute()` des *metrics* toutes les **8 itérations** d'apprentissage afin d'observer les communications des *metrics* pendant l'apprentissage.

</div>

### Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_2.py -b {bs_optim} --image-size {image_size} --test --metric-step 8' 
n_gpu = 4
jobid_metric = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid_metric  = {jobid_metric }')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid_metric)

In [ ]:
dfplot = comm_profiler(jobid_metric, n_display=65)

### Zoom sur les communications

In [ ]:
dfplot = comm_profiler(jobid_metric, n_display=65, zoom=True)

### Valeurs des métriques avant et après les synchronisations

In [ ]:
metric_compute_log(jobid_metric)

### DDP inter-noeud

Nous avons utilisé précédemment **4 GPU** sur le même nœud de calcul. Les bus de communication **intra-nœud** *NVLink* sont très rapide. **Le *scaling* est quasi parfait**.

Si nous utilisons **32 GPU** en *DDP* avec 4 nœuds de calcul et donc des communications sur le réseau d'**interconnexion des nœuds** nous obtenons le résultat suivant.

<div class="alert alert-block alert-danger">

**Ce test n'est pas faisable pendant le TP par chacun d'entre vous, pour des raisons évidentes d'accès aux ressources. Veuillez vous reporter au résultat fourni ici.**

</div>

<img src="./images/ddp32GPU.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-danger">

Assurez-vous que tout se passe bien avant de continuer
    
</div>
<img src="./images/cedez.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-info">

### BatchNorm Layer & SyncBatchNorm Layer
**Rappel** :

Pendant l'apprentissage, la couche normalise ses sorties en utilisant la moyenne et l'écart type du batch d'entrée.
Plus exactement, la couche retourne `(batch - mean(batch)) / (var(batch) + epsilon) * weight + bias` , avec :

* `epsilon`, une petite constante pour éviter la division par 0,
* `weight`, un facteur appris (entraîné) avec un calcul de gradient lors de la backpropagation et qui est initialisé à 1,
* `bias`, un facteur appris (entraîné) avec un calcul de gradient lors de la backpropagation et qui est initialisé à 0.

Pendant l'inférence ou la validation, la couche normalise ses sorties en utilisant en plus des `weight` et `bias` entraînés, les facteurs `running_mean` et `running_var` : `(batch - running_mean) / (running_var + epsilon) * weight + bias`.

`running_mean` et `running_var` sont des facteurs non entraînés, mais qui sont mis à jour à chaque itération de batch lors de l'apprentissage, selon la méthode suivante :

* `running_mean = running_mean * momentum + mean(batch) * (1 - momentum)`
* `running_var = running_var * momentum + var(batch) * (1 - momentum)`

</div>

In [ ]:
import torchvision.models as models
model = models.resnet152()

In [ ]:
BatchNorm_view(jobid, model)

<div class="alert alert-block alert-info">
    
### SyncBatchNorm layer
Voir la [documentation PyTorch](http://www.idris.fr/ia/syncbn.html#syncbn_en_pytorch).

</div>
<div class="alert alert-block alert-warning">

**TODO** : dans le script [dlojz1_2.py](./dlojz1_2.py) :
* Juste avant la bascule du modèle en mode *DistributedDataParallelism*, transformer les couches *BatchNorm* du modèle en couches *SyncBatchNorm*.

</div>

### Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_2.py -b {bs_optim} --image-size {image_size} --test --chkpt'
n_gpu = 4
jobid_sync = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid_sync = {jobid_sync}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid_sync)

In [ ]:
BatchNorm_view(jobid + jobid_sync, model, labels=['BN Layer', 'SyncBN Layers'])

#### Communications

In [ ]:
comm_profiler(jobid_sync, n_display=100)

<div class="alert alert-block alert-danger">

Assurez-vous que tout se passe bien avant de continuer
    
</div>
<img src="./images/cedez.png" style="float: left; margin-right: 1em;"/>